# HMDA. Colorado, 2025


## Why do mortgage applications get abandoned?

### Business question

Around one in five US mortgage applications ends without a decision — the
applicant withdraws, or the file is closed because requested documents never
arrived. Each one is pure cost: the lender has pulled credit, ordered work and
paid staff time, and recovers nothing. The applicant has spent weeks and ended
up without a mortgage.

Unlike a denial, which is a decision, an abandonment is usually friction. Friction
can be fixed.

**Question:** among 2025 Colorado mortgage applications, are applications
submitted through third-party channels more likely to be abandoned before a
decision than those submitted directly to the lender — and does any gap survive
controlling for applicant demographics and loan characteristics?

### What this analysis is about

Nothing here was randomised. Borrowers choose whether to use a broker, and
brokers serve a different mix of borrowers than retail branches do. Any
difference found is an association, not a causal effect. The analysis quantifies
the gap and tests how much of it survives stratification; it does not establish
that channel causes abandonment.

### Data

CFPB HMDA Data Browser, 2025 loan-level records, Colorado, all institutions.
Field definitions from the FFIEC Filing Instructions Guide.

Note: HMDA contains no credit score. It is the strongest predictor in any lending
decision and it is absent by design, which limits what any HMDA-only analysis can
conclude.

### Structure

1. Exploration
2. Defining the outcome
3. Defining the channel variable
4. Hypothesis test
5. Stratified analysis
6. Conclusions

## Exploration 

In [1]:
import pandas as pd
import numpy as np 
from pathlib import Path

In [3]:
raw = Path("../data/raw")
processed = Path ("../data/processed")

file = raw / "state_CO.csv"
df_all = pd.read_csv(file, low_memory = False)

In [4]:
df_all.shape

(295125, 99)

In [7]:
df_all.dtypes.value_counts()

int64      40
float64    32
object     27
Name: count, dtype: int64

In [9]:
def profile_objects(df: pd.DataFrame, n_samples: int = 4) -> pd.DataFrame:
    """Summarise object-dtype columns to spot miscast numerics.

    Returns one row per object column with cardinality, null rate, and a
    sample of values. High cardinality plus numeric-looking samples usually
    means a numeric field contaminated by a sentinel string such as "Exempt".
    """
    objects = df.select_dtypes("object")

    return (
        pd.DataFrame({
            "n_unique": objects.nunique(),
            "percent_null": objects.isna().mean().mul(100).round(1),
            "samples": objects.apply(
                lambda s: list(s.dropna().unique()[:n_samples])
            ),
        })
        .sort_values("n_unique", ascending=False)
    )

In [13]:
profile_objects(df_all)

,n_unique,percent_null,samples
total_loan_costs,97153,51.9,"[0.00, 1690.00, 1790.00, 2104.00]"
origination_charges,61587,51.1,"[0.00, 900.00, 895.00, 13852.50]"
discount_points,52761,74.3,"[12057.50, 5550.00, 13416.82, 30.40]"
loan_to_value_ratio,48122,33.2,"[75, 85, 97, 74.13]"
lender_credits,25689,82.6,"[35.02, 287.50, 4346.40, 4268.88]"
rate_spread,16973,49.5,"[1.312, 0.334, 1.511, 1.01700]"
interest_rate,3554,32.2,"[7.25, 6.375, 5, 5.5]"
lei,1131,0.0,"[549300FGXN1K3HLB1R50, 7H6GLXDRUGQFU57RNE97, 5..."
total_points_and_fees,1011,98.6,"[1690.00, 1790.00, 0.00, Exempt]"
property_value,927,20.3,"[355000, 705000, 195000, 135000]"


In [7]:
df_all["applicant_age"].value_counts(dropna = False, normalize = True)

applicant_age
35-44    0.228141
45-54    0.183173
25-34    0.177078
55-64    0.139856
8888     0.121023
65-74    0.087831
>74      0.038391
<25      0.024508
Name: proportion, dtype: float64

In [14]:
age_known = np.where(sentinel, "age missing", "age present")

for col in ["business_or_commercial_purpose", "action_taken", "derived_sex", "derived_race", "loan_purpose"]:
    print(f"--- {col} ---")
    print(
        pd.crosstab(df_all[col], age_known, normalize="columns")
        .mul(100).round(1)
    )
    print()

--- business_or_commercial_purpose ---
col_0                           age missing  age present
business_or_commercial_purpose                          
1                                      12.2          3.6
2                                      86.8         96.0
1111                                    1.0          0.5

--- action_taken ---
col_0         age missing  age present
action_taken                          
1                     7.6         59.3
2                     0.3          3.8
3                     2.1         15.4
4                     1.4         15.7
5                     0.9          4.5
6                    87.7          0.6
7                     0.0          0.1
8                     0.0          0.6

--- derived_sex ---
col_0              age missing  age present
derived_sex                                
Female                     0.7         20.7
Joint                      1.0         39.2
Male                       1.1         32.4
Sex Not Available      

In [16]:
applications = df_all["action_taken"].isin([1, 2, 3, 4, 5])
still_missing = applications & sentinel

print(still_missing.sum())
print(f"{still_missing.mean() * 100:.1f}% of all rows")
print(f"{still_missing.sum() / applications.sum() * 100:.1f}% of applications")

4404
1.5% of all rows
1.7% of applications


### Cleaning and defining the analysis set

1. **Code labels.** Most HMDA fields are filed as integer codes. Mapping them to
   labels up front means every table and chart downstream is readable.
2. **Dropping non-applications.** `action_taken` codes 6, 7 and 8 are purchased
   loans and preapproval requests. Neither is an application with an applicant
   journey, so both are excluded.
3. **Dropping sentinel ages.** HMDA encodes "not provided" as `8888`. 88% of
   affected rows are purchased loans (already excluded); the 4,404 that remain
   are 1.7% of applications and are dropped so age stays clean downstream.

In [21]:
ACTION_LABELS = {
    1: "originated",
    2: "approved, not accepted",
    3: "denied",
    4: "withdrawn by applicant",
    5: "closed for incompleteness",
    6: "purchased loan",
    7: "preapproval denied",
    8: "preapproval approved, not accepted",
}

LOAN_PURPOSE_LABELS = {
    1: "home purchase",
    2: "home improvement",
    4: "other purpose",
    5: "not applicable",
    31: "refinancing",
    32: "cash-out refinancing",
}

LOAN_TYPE_LABELS = {
    1: "conventional",
    2: "FHA",
    3: "VA",
    4: "RHS or FSA",
}

OCCUPANCY_LABELS = {
    1: "principal residence",
    2: "second residence",
    3: "investment property",
}

LIEN_LABELS = {
    1: "first lien",
    2: "subordinate lien",
}

SUBMISSION_LABELS = {
    1: "submitted directly",
    2: "not submitted directly",
    3: "not applicable",
    1111: "exempt",
}

PAYABLE_LABELS = {
    1: "payable to institution",
    2: "not payable to institution",
    3: "not applicable",
    1111: "exempt",
}

CODE_LABELS = {
    "action_taken": ACTION_LABELS,
    "loan_purpose": LOAN_PURPOSE_LABELS,
    "loan_type": LOAN_TYPE_LABELS,
    "occupancy_type": OCCUPANCY_LABELS,
    "lien_status": LIEN_LABELS,
    "submission_of_application": SUBMISSION_LABELS,
    "initially_payable_to_institution": PAYABLE_LABELS,
}

APPLICATION_ACTIONS = [1, 2, 3, 4, 5]
ABANDONED_ACTIONS = [4, 5]
AGE_SENTINEL = "8888"

In [23]:
def build_analysis_set(df: pd.DataFrame) -> pd.DataFrame:
    """Filter raw HMDA records to analysable applications and add labels.

    Keeps 'action_taken' 1-5 (real applications), drops rows where the
    applicant age is the "not provided" sentinel, adds a binary 'abandoned'
    flag, and maps coded fields to readable labels in new '*_label' columns.
    The original coded columns are kept so results can be checked against
    the filing instructions.
    """
    out = (
        df.loc[df["action_taken"].isin(APPLICATION_ACTIONS)]
        .loc[lambda d: d["applicant_age"].astype(str) != AGE_SENTINEL]
        .copy()
    )

    out["abandoned"] = out["action_taken"].isin(ABANDONED_ACTIONS).astype(int)

    for column, labels in CODE_LABELS.items():
        out[f"{column}_label"] = out[column].map(labels)

    return out


df = build_analysis_set(df_all)

print(f"raw records        {len(df_all):>8,}")
print(f"analysis set       {len(df):>8,}")
print(f"abandonment rate   {df['abandoned'].mean():>8.1%}")

raw records         295,125
analysis set        256,200
abandonment rate      20.5%
